# Hansen Ch.26 Multiple Choice

**Chapter 26 Multiple Choice**（书稿 PDF 约 **p860–861**，习题 **26.1–26.18**）

理论 step-by-step 见同目录 `Hansen_Ch26_Exercises_Solutions.md`。

本 notebook：
1. **CPS** 婚姻状态 multinomial / nested logit（26.12–26.14）
2. **Koppelman** 交通方式：条件 logit / nested / mixed / 简单 MNP（26.15–26.18）

> **写给只学过李子奈/陈强的同学：**
> - MNL：一个人特征 $X$（年龄）→ 各类别一套系数，**相对 base** 解释。
> - Conditional logit：选项特征 $X_j$（价格、时间）→ 共用 $\gamma$，天然适合“选交通方式”。
> - IIA：两选项概率比与第三选项无关——过强时用 nested / mixed / MNP。
> - 报告 **概率剖面与 $\log L$**，不要只读 raw 系数大小。


## 0. 共用工具：条件 logit / 数据装载


In [ ]:
# Hansen Ch.26 — multinomial choice（详尽注释）
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.optimize import minimize
from scipy.stats import norm
from scipy.special import roots_hermitenorm

ROOT = Path("../..") / "hansen" / "econometrics" / "data"  # relative to docs/chXX/
ALTS = ["train", "air", "bus", "car"]  # 0=train 为 base（与 Table 26.1 一致）


def load_koppelman():
    """长表 → 每位旅客一行的 (n,J) 数组。"""
    raw = pd.read_stata(ROOT / "Koppelman" / "Koppelman.dta")
    raw["j"] = raw["alternative"].astype(str).map({a: i for i, a in enumerate(ALTS)})
    raw = raw.sort_values(["case", "j"])
    cases = raw["case"].unique()
    n, J = len(cases), 4
    cost = np.zeros((n, J))
    intime = np.zeros((n, J))
    outtime = np.zeros((n, J))
    income = np.zeros(n)
    urban = np.zeros(n)
    y = np.zeros(n, dtype=int)
    for i, c in enumerate(cases):
        g = raw.loc[raw["case"] == c]
        for _, r in g.iterrows():
            j = int(r["j"])
            cost[i, j] = r["cost"]
            intime[i, j] = r["intime"]
            outtime[i, j] = r["outtime"]
            if r["choice"] == 1:
                y[i] = j
        income[i] = g["income"].iloc[0]
        urban[i] = g["urban"].iloc[0]
    return y, cost, intime, outtime, income, urban


def util_clogit(theta, C, T, income, urban, out=None):
    """
    U_j = gamma' X_j + W' beta_j
    theta: [gc, gt, (go?), air_c,air_inc,air_urb, bus_..., car_...]
    train (j=0) 无 ASC（base）。
    """
    if out is None:
        U = theta[0] * C + theta[1] * T
        idx = 2
    else:
        U = theta[0] * C + theta[1] * T + theta[2] * out
        idx = 3
    for j in [1, 2, 3]:
        c0, bi, bu = theta[idx : idx + 3]
        idx += 3
        U[:, j] = U[:, j] + c0 + bi * income + bu * urban
    return U


def logsumexp_rows(U):
    m = U.max(1, keepdims=True)
    return (m.squeeze() + np.log(np.exp(U - m).sum(1))), np.exp(U - m) / np.exp(U - m).sum(1, keepdims=True)


def fit_clogit(y, C, T, income, urban, out=None, x0=None):
    n = len(y)
    if out is None:
        if x0 is None:
            x0 = np.array([-0.022, -0.015, -2.15, 0.036, 0.29, -1.79, -0.051, -0.23, 1.86, 0.008, -0.99])
        def nll(th):
            U = util_clogit(th, C, T, income, urban)
            ls, _ = logsumexp_rows(U)
            return -(U[np.arange(n), y] - ls).sum()
    else:
        if x0 is None:
            x0 = np.array([-0.015, -0.018, -0.03, -2, 0.03, 0.3, -1.5, -0.05, -0.2, 1.5, 0.01, -1])
        def nll(th):
            U = util_clogit(th, C, T, income, urban, out=out)
            ls, _ = logsumexp_rows(U)
            return -(U[np.arange(n), y] - ls).sum()

    res = minimize(nll, x0, method="BFGS", options={"maxiter": 400})
    th = res.x
    # BHHH 标准误：score 外积
    U = util_clogit(th, C, T, income, urban, out=out)
    _, P = logsumexp_rows(U)
    S = np.zeros((n, len(th)))
    for i in range(n):
        for j in range(4):
            d = (1.0 if y[i] == j else 0.0) - P[i, j]
            S[i, 0] += d * C[i, j]
            S[i, 1] += d * T[i, j]
            base0 = 2
            if out is not None:
                S[i, 2] += d * out[i, j]
                base0 = 3
            if j >= 1:
                b = base0 + 3 * (j - 1)
                S[i, b] += d
                S[i, b + 1] += d * income[i]
                S[i, b + 2] += d * urban[i]
    se = np.sqrt(np.maximum(np.diag(np.linalg.inv(S.T @ S)), 0))
    return th, se, -res.fun


print("helpers ready")


## 26.12–26.13　CPS 婚姻 multinomial logit


In [ ]:
# ---- 婚姻四分类 MNL ----
cps = pd.read_stata(ROOT / "cps09mar" / "cps09mar.dta")

def marital4(s):
    """married(含丧偶)/divorced/separated/never — 与正文 Figure 26.1 口径一致。"""
    m = s.values
    y = np.full(len(m), -1)
    y[np.isin(m, [1, 2, 3, 4])] = 0
    y[m == 5] = 1
    y[m == 6] = 2
    y[m == 7] = 3  # base in estimation
    return y

LABELS = ["married", "divorced", "separated", "never"]


def mnl_fit(y, X, base=3):
    """相对 base 的 (J-1)*k 系数；softmax MLE。"""
    n, k = X.shape
    J = int(y.max()) + 1
    others = [j for j in range(J) if j != base]

    def nll(th):
        B = np.zeros((J, k))
        for i, j in enumerate(others):
            B[j] = th[i * k : (i + 1) * k]
        V = X @ B.T
        m = V.max(1, keepdims=True)
        ls = m.squeeze() + np.log(np.exp(V - m).sum(1))
        return -(V[np.arange(n), y] - ls).sum()

    res = minimize(nll, np.zeros((J - 1) * k), method="BFGS", options={"maxiter": 400})
    th = res.x
    B = np.zeros((J, k))
    for i, j in enumerate(others):
        B[j] = th[i * k : (i + 1) * k]
    return B, -res.fun


def predict_P(B, X):
    V = X @ B.T
    m = V.max(1, keepdims=True)
    e = np.exp(V - m)
    return e / e.sum(1, keepdims=True)


# 26.12 男性 ~ age + age^2/100
men = cps.loc[cps["female"] == 0].copy()
y = marital4(men["marital"])
mask = y >= 0
age = men.loc[mask, "age"].values.astype(float)
y = y[mask]
X = np.column_stack([np.ones(len(y)), age, age ** 2 / 100])
Bm, llm = mnl_fit(y, X)
print(f"26.12 men n={len(y)}  logL={llm:.1f}")
ages = np.array([25, 35, 45, 55, 65, 75])
Xp = np.column_stack([np.ones(len(ages)), ages, ages ** 2 / 100])
Pm = predict_P(Bm, Xp)
print("age", ages)
for j, lab in enumerate(LABELS):
    print(f"  {lab:10s}", np.round(Pm[:, j], 3))

# 对照：大学女性
cw = cps.loc[(cps["female"] == 1) & (cps["education"] >= 16)].copy()
yw = marital4(cw["marital"]); m = yw >= 0
agew = cw.loc[m, "age"].values.astype(float); yw = yw[m]
Xw = np.column_stack([np.ones(len(yw)), agew, agew ** 2 / 100])
Bw, llw = mnl_fit(yw, Xw)
Pw = predict_P(Bw, Xp)
print(f"\ncompare college women n={len(yw)} logL={llw:.1f}")
for j, lab in enumerate(LABELS):
    print(f"  {lab:10s}", np.round(Pw[:, j], 3))

# 26.13 女 age<=35 ~ age + education
w = cps.loc[(cps["female"] == 1) & (cps["age"] <= 35)].copy()
y = marital4(w["marital"]); m = y >= 0
w = w.loc[m]; y = y[m]
X = np.column_stack([np.ones(len(y)), w["age"].values, w["education"].values])
B, ll = mnl_fit(y, X)
print(f"\n26.13 women age<=35 n={len(y)} logL={ll:.1f}")
for j, lab in enumerate(LABELS):
    print(lab, "const,age,educ =", np.round(B[j], 4))
for e in [12, 16, 18]:
    Xp = np.column_stack([[1.0], [w["age"].mean()], [e]])
    print(f"  educ={e}: P=", np.round(predict_P(B, Xp)[0], 3))


## 26.14　女性婚姻 nested logit（分组说明见 md）

分组：`{divorced, separated}` 同 nest（曾婚破裂的相近状态）；`married`、`never` 单点 nest（$\tau=1$）。


In [ ]:
# ---- nested logit on marital status (women, age only) ----
wom = cps.loc[cps["female"] == 1].copy()
y = marital4(wom["marital"]); m = y >= 0
age = wom.loc[m, "age"].values.astype(float); y = y[m]
n = len(y)

def nested_marital_nll(th):
    # th: (const,age) for married, div, sep + tau for {div,sep}
    tau = th[6]
    if not (0.05 < tau <= 1.0):
        return 1e12
    U = np.zeros((n, 4))
    U[:, 0] = th[0] + th[1] * age
    U[:, 1] = th[2] + th[3] * age
    U[:, 2] = th[4] + th[5] * age
    groups = [[0], [1, 2], [3]]
    taus = [1.0, tau, 1.0]
    I, Pc = [], {}
    for g, mem in enumerate(groups):
        t = taus[g]
        Um = U[:, mem] / t
        mx = Um.max(1, keepdims=True)
        Ig = (np.exp(mx) * np.exp(Um - mx).sum(1, keepdims=True)).squeeze()
        I.append(np.maximum(Ig, 1e-300))
        for li, j in enumerate(mem):
            Pc[j] = np.exp(Um[:, li] - np.log(I[g]))
    logIt = [taus[g] * np.log(I[g]) for g in range(3)]
    st = np.column_stack(logIt)
    mx = st.max(1, keepdims=True)
    ls = mx.squeeze() + np.log(np.exp(st - mx).sum(1))
    Pg = [np.exp(logIt[g] - ls) for g in range(3)]
    P = np.zeros((n, 4))
    for g, mem in enumerate(groups):
        for j in mem:
            P[:, j] = Pc[j] * Pg[g]
    return -np.log(np.clip(P[np.arange(n), y], 1e-300, None)).sum()

res = minimize(
    nested_marital_nll,
    np.array([1.0, 0.02, -1.0, 0.01, -1.5, 0.0, 0.7]),
    method="L-BFGS-B",
    bounds=[(None, None)] * 6 + [(0.05, 1.0)],
    options={"maxiter": 250},
)
print(f"26.14 nested women logL={-res.fun:.1f}, tau{{div,sep}}={res.x[6]:.3f}")
print("params (c0,a0,c1,a1,c2,a2,tau)=", np.round(res.x, 4))


## 26.15　Koppelman 条件 logit


In [ ]:
y, cost, intime, outtime, income, urban = load_koppelman()
print("Koppelman n=", len(y), "shares=", {ALTS[j]: (y == j).mean() for j in range(4)})

# (a) baseline Table 26.1
th, se, ll = fit_clogit(y, cost, intime, income, urban)
print(f"\n(a) baseline  logL={ll:.2f}  [table -2100.6]")
print(f"  cost   {th[0]:.4f} ({se[0]:.4f})   table -0.022 (0.003)")
print(f"  intime {th[1]:.4f} ({se[1]:.4f})   table -0.015 (0.001)")

# (b) + outtime
th, se, ll = fit_clogit(y, cost, intime, income, urban, out=outtime)
print(f"\n(b) +outtime logL={ll:.2f}")
print(f"  cost {th[0]:.4f}, intime {th[1]:.4f}, outtime {th[2]:.4f}")

# (c) total time
th, se, ll = fit_clogit(y, cost, intime + outtime, income, urban)
print(f"\n(c) time=in+out logL={ll:.2f}")
print(f"  cost {th[0]:.4f}, time {th[1]:.4f}")

# (d) logs
th, se, ll = fit_clogit(
    y, np.log(cost), np.log(intime), income, urban,
    x0=np.array([-1, -1.5, -2, 0.03, 0.3, -1.5, -0.05, -0.2, 1.5, 0.01, -1]),
)
print(f"\n(d) log cost/time logL={ll:.2f}")
print(f"  logcost {th[0]:.4f}, logintime {th[1]:.4f}")


## 26.16　嵌套 logit

默认：`{train,bus}` $\tau=1$（约束），`{air,car}` 自由 $\tau$。


In [ ]:
def nested_nll(th, y, C, T, income, urban, group_defs, tau_fixed):
    """th = 11 utility params + free taus；定理 26.2 的 P_jk = P(k|j) P(j)。"""
    nU = 11
    thU = th[:nU]
    n_g = len(group_defs)
    free = [g for g in range(n_g) if g not in tau_fixed]
    taus = np.ones(n_g)
    for i, gi in enumerate(free):
        taus[gi] = th[nU + i]
    for gi, v in tau_fixed.items():
        taus[gi] = v
    if np.any(taus <= 0.02):
        return 1e12
    U = util_clogit(thU, C, T, income, urban)
    n, J = U.shape
    I_list, Pcond = [], {}
    for g, members in enumerate(group_defs):
        tau = taus[g]
        Um = U[:, members] / tau
        m = Um.max(1, keepdims=True)
        Ig = (np.exp(m) * np.exp(Um - m).sum(1, keepdims=True)).squeeze()
        I_list.append(np.maximum(Ig, 1e-300))
        for li, j in enumerate(members):
            Pcond[j] = np.exp(Um[:, li] - np.log(I_list[g]))
    logI_tau = [taus[g] * np.log(I_list[g]) for g in range(n_g)]
    stack = np.column_stack(logI_tau)
    m = stack.max(1, keepdims=True)
    logsum = m.squeeze() + np.log(np.exp(stack - m).sum(1))
    Pg = [np.exp(logI_tau[g] - logsum) for g in range(n_g)]
    Pjks = np.zeros((n, J))
    for g, members in enumerate(group_defs):
        for j in members:
            Pjks[:, j] = Pcond[j] * Pg[g]
    return -np.log(np.clip(Pjks[np.arange(n), y], 1e-300, None)).sum()


def fit_nested(y, C, T, income, urban, group_defs, tau_fixed, x0=None):
    free = [g for g in range(len(group_defs)) if g not in tau_fixed]
    if x0 is None:
        x0 = np.array([-0.011, -0.005, -0.46, 0.024, 0.28, -1.55, -0.049, -0.21, 1.19, 0.017, -0.58]
                      + [0.24] * len(free))
    res = minimize(
        lambda th: nested_nll(th, y, C, T, income, urban, group_defs, tau_fixed),
        x0, method="L-BFGS-B",
        bounds=[(None, None)] * 11 + [(0.05, 1.0)] * len(free),
        options={"maxiter": 400, "ftol": 1e-12},
    )
    return res.x, -res.fun


# (a) Table 26.1 nesting
th, ll = fit_nested(y, cost, intime, income, urban, [[0, 2], [1, 3]], {0: 1.0})
print(f"(a) {{train,bus}}|{{air,car}}  logL={ll:.2f}  [table -2044.4]")
print(f"  cost={th[0]:.4f}, intime={th[1]:.4f}, tau_air_car={th[11]:.3f}  [table tau=0.24]")

# (b) logs
th, ll = fit_nested(
    y, np.log(cost), np.log(intime), income, urban, [[0, 2], [1, 3]], {0: 1.0},
    x0=np.array([-0.5, -0.5, -0.5, 0.02, 0.3, -1.5, -0.05, -0.2, 1.2, 0.02, -0.6, 0.3]),
)
print(f"(b) log vars  logL={ll:.2f}, tau={th[11]:.3f}")

# (c) {car} vs {train,bus,air}
th, ll = fit_nested(
    y, cost, intime, income, urban, [[3], [0, 1, 2]], {0: 1.0},
    x0=np.array([-0.02, -0.01, -2, 0.03, 0.3, -1.5, -0.05, -0.2, 1.5, 0.01, -1, 0.5]),
)
print(f"(c) {{car}}|{{public+air}} logL={ll:.2f}, tau_public={th[11]:.3f}")

# (d) {air} vs {train,bus,car}
th, ll = fit_nested(
    y, cost, intime, income, urban, [[1], [0, 2, 3]], {0: 1.0},
    x0=np.array([-0.02, -0.01, -2, 0.03, 0.3, -1.5, -0.05, -0.2, 1.5, 0.01, -1, 0.5]),
)
print(f"(d) {{air}}|{{ground}} logL={ll:.2f}, tau_ground={th[11]:.3f}")
print("→ (d) 接近条件 logit：地面方式并非高度相关的一类。")


## 26.17　混合 logit（intime 随机系数）


In [ ]:
# 模拟积分：eta ~ N(mu, sig^2) 或 -lognormal
G = 80
rng = np.random.default_rng(42)
Z = rng.standard_normal(G)


def mixed_nll(th, y, C, T, income, urban, Z, lognormal=False):
    gc, mu, sig = th[0], th[1], abs(th[2]) + 1e-8
    rest = th[3:]
    U0 = gc * C
    idx = 0
    for j in [1, 2, 3]:
        c0, bi, bu = rest[idx : idx + 3]
        idx += 3
        U0[:, j] = U0[:, j] + c0 + bi * income + bu * urban
    coef = (-np.exp(mu + sig * Z)) if lognormal else (mu + sig * Z)
    acc = np.zeros(len(y))
    n = len(y)
    for g in range(len(Z)):
        U = U0 + coef[g] * T
        ls, _ = logsumexp_rows(U)
        acc += np.exp(U[np.arange(n), y] - ls)
    return -np.log(np.clip(acc / len(Z), 1e-300, None)).sum()


def fit_mixed(C, T, lognormal=False):
    if lognormal:
        x0 = np.array([-0.023, np.log(0.014), 0.3, -2.7, 0.04, 0.35, -1.8, -0.05, -0.24, 1.9, 0.008, -1.0])
    else:
        x0 = np.array([-0.023, -0.014, 0.005, -2.7, 0.04, 0.35, -1.8, -0.05, -0.24, 1.9, 0.008, -1.0])
    res = minimize(
        lambda th: mixed_nll(th, y, C, T, income, urban, Z, lognormal),
        x0, method="L-BFGS-B", options={"maxiter": 120},
    )
    return res.x, -res.fun


th, ll = fit_mixed(cost, intime)
print(f"(a) normal intime  logL={ll:.2f}  [table -2095.5]")
print(f"  cost={th[0]:.4f}, E[eta]={th[1]:.4f}, sigma={abs(th[2]):.4f}  [table -0.023,-0.014,0.0048]")

th, ll = fit_mixed(cost, intime + outtime)
print(f"(b) total time     logL={ll:.2f}")
print(f"  cost={th[0]:.4f}, E[eta]={th[1]:.4f}, sigma={abs(th[2]):.4f}")

th, ll = fit_mixed(cost, intime, lognormal=True)
Eeta = -np.exp(th[1] + 0.5 * th[2] ** 2)
print(f"(c) lognormal |time| logL={ll:.2f}, E[eta]≈{Eeta:.4f}")


## 26.18　简单多项 probit（Gauss–Hermite）

一般相关 MNP 需 GHK，计算重；此处复现 Table 26.1 **Simple Multi. Probit**（独立正态误差）。


In [ ]:
nodes, weights = roots_hermitenorm(16)


def simple_mnp_ll(theta, y, C, T, income, urban):
    """Theorem 26.3：P_j = ∫ ∏_{ℓ≠j} Φ(μ_j−μ_ℓ+v) φ(v) dv。"""
    U = util_clogit(theta, C, T, income, urban)
    n, J = U.shape
    P = np.zeros((n, J))
    for j in range(J):
        acc = np.zeros(n)
        for v, w in zip(nodes, weights):
            prod = np.ones(n)
            for ell in range(J):
                if ell == j:
                    continue
                prod *= norm.cdf(U[:, j] - U[:, ell] + v)
            acc += w * prod
        P[:, j] = acc / np.sqrt(2 * np.pi)
    P = np.clip(P, 1e-300, None)
    P = P / P.sum(1, keepdims=True)
    return np.log(P[np.arange(n), y]).sum()


def fit_smnp(C, T, x0=None):
    if x0 is None:
        x0 = np.array([-0.018, -0.011, -1.51, 0.027, 0.29, -1.45, -0.019, -0.13, 1.44, 0.006, -0.73])
    res = minimize(
        lambda th: -simple_mnp_ll(th, y, C, T, income, urban),
        x0, method="BFGS", options={"maxiter": 60},
    )
    return res.x, simple_mnp_ll(res.x, y, C, T, income, urban)


th, ll = fit_smnp(cost, intime)
print(f"(a) simple MNP  logL={ll:.2f}  [table -2109.3]")
print(f"  cost={th[0]:.4f}, intime={th[1]:.4f}  [table -0.018, -0.011]")

th, ll = fit_smnp(
    np.log(cost), np.log(intime),
    x0=np.array([-1, -1.5, -1.5, 0.03, 0.3, -1.4, -0.02, -0.1, 1.4, 0.006, -0.7]),
)
print(f"(b) log vars     logL={ll:.2f}")
print(f"  logcost={th[0]:.4f}, logintime={th[1]:.4f}")
print("""
一般 MNP（表 26.1 末列）：logL=-2017.4，Corr(air,car)≈0.99，
与 nested 的 air–car 高相关一致；边际效应幅度小于条件 logit。
""")
